# Outline

> Rounded rectilinear outlines for groups of grid cells, and their conversion to matplotlib paths.

In [ ]:
#| default_exp core.outline

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

Every ItemiSet figure draws groups of grid cells as one shape: the union of the cells, shrunk slightly so neighbouring or nested outlines do not sit on top of each other, with rounded corners.

In [ ]:
#| export
from matplotlib.path import Path as MplPath
from shapely.geometry import MultiPolygon, Polygon, box
from shapely.ops import unary_union

## Rounded regions

In [ ]:
#| export
def rounded_region(boxes, inset: float, radius: float):
    "Union of `boxes`, shrunk by `inset` (square corners), then rounded: inner corners first, then outer ones, both with `radius`."
    p = unary_union(boxes).buffer(0)
    p = p.buffer(-inset, join_style="mitre")
    p = p.buffer(radius, join_style="round").buffer(-radius, join_style="round")                  # round inner corners
    p = p.buffer(-radius * 0.999, join_style="round").buffer(radius * 0.999, join_style="round")  # round outer corners
    return p

An L of three unit cells, shrunk by 0.1 with corners of radius 0.2, is one polygon that lies inside the cells, keeps their overall shape, and has lost a little area to the inset and the rounding:

In [ ]:
cells = [box(0, 0, 1, 1), box(1, 0, 2, 1), box(0, 1, 1, 2)]
L = rounded_region(cells, inset=0.1, radius=0.2)
test_eq(L.geom_type, "Polygon")
assert L.within(unary_union(cells))
test_close(L.bounds, (0.1, 0.1, 1.9, 1.9), eps=1e-6)
assert 2.0 < L.area < 3 - 2 * 0.1 * 3

## Drawing shapes with matplotlib

In [ ]:
#| export
def geoms(g) -> list:
    "The polygons in a shapely geometry, flattening collections and skipping points and lines."
    if g.is_empty:
        return []
    if isinstance(g, Polygon):
        return [g]
    if isinstance(g, MultiPolygon):
        return list(g.geoms)
    return [p for p in getattr(g, "geoms", []) if isinstance(p, (Polygon, MultiPolygon))
            for p in geoms(p)]

def poly_to_path(geom):
    "A matplotlib `Path` for a shapely (multi)polygon, holes included, or `None` if it is empty."
    verts, codes = [], []
    for p in geoms(geom):
        for ring in [p.exterior, *p.interiors]:
            xy = list(ring.coords)
            verts += xy
            codes += [MplPath.MOVETO] + [MplPath.LINETO] * (len(xy) - 2) + [MplPath.CLOSEPOLY]
    return MplPath(verts, codes) if verts else None

In [ ]:
test_eq(geoms(L), [L])
test_eq(geoms(L.difference(L)), [])
two = unary_union([box(0, 0, 1, 1), box(3, 0, 4, 1)])
test_eq(len(geoms(two)), 2)
test_eq(poly_to_path(L.difference(L)), None)
test_eq(len(poly_to_path(two).vertices), 10)  # two closed squares, 5 vertices each

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()